## 13. Build the end-to-end Demo Pipeline

This block wraps the main recommendation workflow into one reusable function called `run_demo()`. Instead of running the evidence extraction, disease similarity retrieval, reranking, graph visualization, and explanation steps separately, this section combines them into a single callable pipeline.

The purpose of this block is to make the notebook easier to use as a demonstration tool. Once this function is defined, the user only needs to provide:
- a disease query
- a model choice (`"hgt"` or `"baseline"`)

At a high level, `run_demo()` generates four results in sequence:

1. a **top 5 drug candidate table**
2. a **top 10 similar diseases table**
3. an **evidence and similarity subgraph**
4. a **human-readable RAG-style explanation report**

### 1. Validate dependencies and import display utilities

The block begins by importing the libraries needed for formatting, plotting, graph drawing, and report generation. This includes:
- `json`
- `textwrap`
- `numpy`
- `pandas`
- `torch`
- `networkx`
- `matplotlib`

It then performs a series of guard checks to make sure the required functions and lookup tables from earlier sections are already available. In particular, this block depends on:
- the evidence extraction pipeline from **Block 10**
- the disease matching and treatment lookup functions from **Block 11**
- the disease similarity helpers from **Block 12**

### 2. Define shared text-formatting helpers

Before defining the main demo logic, the block creates several small helper functions that make the final outputs easier to read.

These include utilities for:
- wrapping long labels so they fit inside graph nodes
- safely converting missing values to readable strings
- formatting numeric values to a fixed number of decimals
- building bullet lists for report text
- assigning display priority to evidence types
- converting raw evidence rows into more natural human-readable text
- converting safety rows into readable warning sentences
- translating a risk level such as `low`, `medium`, or `high` into a full sentence


### 3. Build the demo subgraph

The next major helper is `_build_demo_subgraph(...)`.

This function creates a **NetworkX directed graph** that combines several types of information into one visual explanation. The graph is centered on the selected query disease and includes three main reasoning layers:

- **candidate drug predictions**
- **supporting evidence paths**
- **similar disease links**

It also includes **safety flag nodes** attached to the relevant drug candidates.

The graph is built with the following node groups:

- **query disease node** at the center/right
- **top candidate drug nodes** on the left
- **protein evidence nodes** between drugs and the disease
- **phenotype or effect nodes** between drugs and the disease
- **similar disease nodes** near the query disease
- **safety nodes** hanging off the candidate drugs

The function adds different kinds of edges depending on their role:

- **prediction edges** from drug to query disease
- **evidence edges** through proteins or phenotypes
- **similarity edges** from similar diseases to the query disease
- **safety edges** from a drug to a warning node

### 4. Draw the demo subgraph

After building the graph object, the block defines `_draw_demo_subgraph(...)` to render it.

This function uses a layered layout so the output stays readable. Each node type is placed in its own vertical column:

- candidate drugs
- protein evidence
- phenotype evidence
- similar diseases
- query disease

Safety nodes are positioned near the candidate drugs.


### 5. Build the RAG-style explanation report

The next major helper is `_build_rag_report(...)`.

This function converts the structured recommendation outputs into a human-readable explanation report. It summarizes:
- the selected disease
- the most similar diseases
- the top recommended drugs
- the evidence supporting each drug
- the safety concerns associated with each drug
- the similar-disease support behind each recommendation

The report is designed in two formats at the same time:

#### Human-readable report

This is a narrative-style summary that can be printed directly in the notebook. It helps explain the recommendation results in plain language rather than only through tables.

#### Structured RAG report table

The function also creates a dataframe version of the explanation so each recommendation can be inspected in a more structured format.

#### Prompt packet for downstream generation

In addition, the function prepares a JSON-like prompt packet that can be reused in a retrieval-augmented generation workflow or passed into a language model for richer explanation generation later.

This makes the block useful not only for notebook display, but also for downstream reporting or demo applications.

### 6. Define the `run_demo()` pipeline

The main function in this block is `run_demo(disease_query, model_choice="hgt")`.

#### Step A: Select the model

The function first checks the chosen model type:
- if `model_choice="hgt"`, it uses the HGT model from Block 8
- if `model_choice="baseline"`, it uses the baseline heterogeneous GNN from Block 7

This allows the same demo pipeline to be reused for both models without rewriting any of the downstream logic.

#### Step B: Match the disease query

Next, the function resolves the free-text disease query using `match_disease(...)` from Block 11.

The best disease match is selected as the active disease for the rest of the pipeline. The matched disease name and disease index are stored for later use.

#### Step C: Retrieve and rerank candidate drugs

The function then scores all drugs for the selected disease using the chosen model. It excludes drugs already known as training indications for that disease, keeps a candidate pool, and extracts an evidence bundle for each drug.

At this stage, the function computes:
- a normalized evidence score
- a normalized safety penalty
- an initial rerank score based on raw model score, evidence, and safety

#### Step D: Compute disease similarity

After building the candidate pool, the function computes similarity between the query disease and all other diseases. It uses the same multi-signal idea from Block 12, combining:
- embedding cosine similarity
- direct disease–disease connections
- shared proteins
- PPI-expanded overlap
- shared effects
- overlap in known therapeutic drugs

These components are combined into a single similarity score, and the top similar diseases are retained in a ranked table.

#### Step E: Compute similar-disease support for each drug

The function then asks whether each candidate drug is already known to treat any of the retrieved similar diseases.

If so, the drug receives a **similar disease support** boost. The score is normalized and added as a new reranking term.

This extends the ranking from:
- model score
- graph evidence
- safety penalty

to:
- model score
- graph evidence
- similar-disease support
- safety penalty

The final top 5 candidates are stored in a dataframe for display.

#### Step F: Build the detailed result objects

For each of the final top candidates, the function also builds a detailed result record containing:
- drug metadata
- reranking scores
- evidence tables
- safety tables
- similar-disease support
- JSON-ready explanation content

These records are stored in a list that later feeds both the subgraph and the RAG report.

### 7. Display the four final demo outputs

After all calculations are complete, the function displays four outputs in order.

#### Output 1: Top 5 candidate drugs

This table shows the final recommended drugs for the selected disease. It includes:
- drug name
- raw model score
- similar disease support
- final rerank score
- risk level
- number of evidence items
- number of safety flags
- known treatment history

This is the main recommendation table the user sees first.

#### Output 2: Top 10 similar diseases

This table shows the diseases most similar to the selected query disease, along with the main similarity statistics such as:
- combined similarity score
- embedding similarity
- shared proteins
- shared effects
- shared drugs

This helps explain why the disease-aware reranking step boosted certain candidates.

#### Output 3: Evidence and similarity subgraph

The function then builds and draws the layered explanation graph described earlier. This gives a compact visual summary of:
- recommended drugs
- biological evidence paths
- similar diseases
- safety warnings

#### Output 4: RAG explanation and summary

Finally, the function prints the narrative explanation report and displays the structured report table. This serves as the most detailed interpretation layer in the whole notebook.

### 8. Return all artifacts for reuse

Even though the demo function displays everything directly, it also returns a dictionary containing all key result objects. These include:
- the matched disease
- the selected model name
- the disease match candidates
- the similar disease table
- the final recommendation table
- the detailed recommendation records
- the NetworkX subgraph
- the structured RAG dataframe
- the final human-readable report
- the prompt packet for downstream generation

Returning these objects makes the block reusable for additional analysis beyond the notebook display itself.

### Why this block matters

This block is important because it turns the notebook from a collection of separate analysis stages into a single end-to-end demonstration pipeline. The earlier blocks developed the individual pieces, but this section combines them into one function that can be reused easily for any disease query.

It matters because it:
- integrates evidence extraction, disease similarity, and reranking into one pipeline
- produces both visual and textual explanations
- supports either the baseline model or the HGT model
- returns structured artifacts for further analysis
- prepares the notebook for the final user-facing demo in Block 14

In short, this block is the orchestration layer of the project. It connects the trained models, graph evidence logic, disease similarity retrieval, and explanation generation into one complete drug repurposing demo.

In [ ]:
# DEMO / OUTPUT Function Block

# Guards
assert "extract_evidence_bundle"      in globals(), "Run Block 10 first."
assert "score_all_drugs_for_disease"  in globals(), "Run Block 10 first."
assert "TRAIN_THERAPEUTIC_BY_DISEASE" in globals(), "Run Block 10 first."
assert "match_disease"                in globals(), "Run Block 11 first."
assert "get_known_treatments"         in globals(), "Run Block 11 first."
assert "_build_sets"                  in globals(), "Run Block 12 first."
assert "_ppi_expand"                  in globals(), "Run Block 12 first."
assert "_log_norm"                    in globals(), "Run Block 12 first."



# Shared Text Helpers


def _wrap(s, w=100):
    return "\n".join(textwrap.wrap(str(s), w))

def _safe(x):
    return "" if (x is None or (isinstance(x, float) and np.isnan(x))) else str(x)

def _fmt(x, n=3):
    try:    return f"{float(x):.{n}f}"
    except: return str(x)

def _bullets(lines, prefix="  • "):
    return "\n".join(prefix + str(l).strip() for l in lines if str(l).strip())

def _ev_priority(ev_type):
    return {"shared_protein_bridge": 1, "ppi_bridge": 2,
            "disease_disease_transfer": 3, "effect_or_phenotype_overlap": 4}.get(str(ev_type), 99)

def _humanise_evidence(row):
    t    = _safe(row.get("evidence_type", ""))
    path = _safe(row.get("path_text", ""))
    labels = {
        "shared_protein_bridge":      "Shared protein target",
        "ppi_bridge":                 "Protein interaction bridge",
        "disease_disease_transfer":   "Related disease transfer",
        "effect_or_phenotype_overlap":"Phenotype / effect overlap",
    }
    return f"{labels.get(t, 'Graph evidence')}: {path}"

def _humanise_safety(row):
    t   = _safe(row.get("safety_type", ""))
    txt = _safe(row.get("text", ""))
    labels = {
        "direct_contraindication": "Direct safety warning",
        "adverse_effect_overlap":  "Potential adverse overlap",
    }
    prefix = labels.get(t, "Safety signal")
    return f"{prefix}: {txt}" if txt else f"{prefix}: {t}"

def _risk_sentence(risk_level):
    return {
        "low":    "low observed graph-based safety concern",
        "medium": "moderate observed graph-based safety concern",
        "high":   "high observed graph-based safety concern",
    }.get(str(risk_level).lower(), "unspecified graph-based safety concern")


# Subgraph builder

def _build_demo_subgraph(
    dis_idx:         int,
    dis_name:        str,
    detailed_results:list,
    similar_diseases:pd.DataFrame,
    top_k_drugs:     int = 5,
    top_k_similar:   int = 5,
) -> nx.DiGraph:
    """
    Build a NetworkX DiGraph that integrates three evidence layers:
      1. Drug → query disease prediction edges (solid blue)
      2. Drug → protein/phenotype → disease evidence paths (solid green)
      3. Similar disease → query disease similarity edges (dashed purple)
      4. Drug → safety flag nodes (dashed red)

    Layout columns (left to right):
      Drugs | Evidence nodes | Similar diseases | Query disease
    """

    def uid(*parts):
        return "|".join(str(p) for p in parts)

    G = nx.DiGraph()

    # Centre: query disease
    dis_uid = uid("qd", dis_idx)
    G.add_node(dis_uid,
               label=_wrap(dis_name, 20),
               role="query_disease",
               full_name=dis_name)

    # Left: drug nodes + prediction + evidence paths
    for item in detailed_results[:top_k_drugs]:
        drug_idx  = item["drug_idx"]
        drug_uid  = uid("drug", drug_idx)
        score     = item["rerank_score"]
        risk      = item["risk_level"]

        G.add_node(drug_uid,
                   label=_wrap(f"#{item['rank']} {item['drug_name']}", 22),
                   role="drug",
                   full_name=item["drug_name"],
                   pred_score=score,
                   risk_level=risk,
                   known_for=item.get("known_treatment_for", ""))

        # Prediction edge — drug → query disease
        G.add_edge(drug_uid, dis_uid,
                   role="prediction",
                   weight=score,
                   label=f"score={_fmt(score)}")

        ev_df = item["evidence_df"]
        sf_df = item["safety_df"]

        # Evidence paths — drug → intermediate → query disease
        for ev_i, ev in ev_df.iterrows():
            ev_type = str(ev.get("evidence_type", ""))

            if ev_type in {"shared_protein_bridge", "ppi_bridge"}:
                mid_uid = uid("prot", ev.get("intermediate_idx", ev_i), drug_idx)
                if not G.has_node(mid_uid):
                    G.add_node(mid_uid,
                               label=_wrap(str(ev.get("intermediate_name", "protein")), 18),
                               role="protein",
                               full_name=str(ev.get("intermediate_name", "")))
                G.add_edge(drug_uid, mid_uid, role="evidence", label=ev_type)
                G.add_edge(mid_uid, dis_uid,  role="evidence", label=ev_type)

            elif ev_type == "effect_or_phenotype_overlap":
                mid_uid = uid("pheno", ev.get("intermediate_idx", ev_i), drug_idx)
                if not G.has_node(mid_uid):
                    G.add_node(mid_uid,
                               label=_wrap(str(ev.get("intermediate_name", "phenotype")), 18),
                               role="phenotype",
                               full_name=str(ev.get("intermediate_name", "")))
                G.add_edge(drug_uid, mid_uid, role="evidence", label=ev_type)
                G.add_edge(mid_uid, dis_uid,  role="evidence", label=ev_type)

            elif ev_type == "disease_disease_transfer":
                pass

        # Safety flag nodes
        for sf_i, sf in sf_df.iterrows():
            sf_uid = uid("safety", drug_idx, sf_i)
            sev    = str(sf.get("severity", ""))
            G.add_node(sf_uid,
                       label=_wrap(str(sf.get("safety_type", "safety")).replace("_", " "), 20),
                       role="safety",
                       full_name=str(sf.get("text", "")),
                       severity=sev)
            G.add_edge(drug_uid, sf_uid, role="safety", label=sev)

    # Centre-right: similar disease nodes
    for _, srow in similar_diseases.head(top_k_similar).iterrows():
        sim_idx   = int(srow["similar_disease_idx"])
        sim_name  = str(srow["similar_disease_name"])
        sim_score = float(srow["combined_similarity_score"])
        sim_uid   = uid("simdis", sim_idx)

        if not G.has_node(sim_uid):
            G.add_node(sim_uid,
                       label=_wrap(sim_name, 20),
                       role="similar_disease",
                       full_name=sim_name,
                       sim_score=sim_score)

        # Similarity edge — similar disease → query disease
        G.add_edge(sim_uid, dis_uid,
                   role="similarity",
                   weight=sim_score,
                   label=f"sim={_fmt(sim_score)}")

    return G


def _draw_demo_subgraph(
    G:        nx.DiGraph,
    dis_name: str,
    model_name: str,
    save_path: str = "/mnt/data/demo_subgraph.png",
):
    """
    Draw the demo subgraph with a clean 4-column layered layout.

    Layout:
      x=0.05  Drugs
      x=0.35  Protein evidence nodes
      x=0.55  Phenotype / effect nodes
      x=0.72  Similar diseases
      x=0.95  Query disease (centre)
    """
    ROLE_X = {
        "drug":            0.05,
        "protein":         0.35,
        "phenotype":       0.55,
        "similar_disease": 0.72,
        "safety":          0.18,
        "query_disease":   0.95,
    }
    ROLE_COLOR = {
        "drug":            "#8ecae6",
        "protein":         "#ffb703",
        "phenotype":       "#bde0fe",
        "similar_disease": "#cdb4db",
        "safety":          "#ef476f",
        "query_disease":   "#fb8500",
    }
    ROLE_SIZE = {
        "drug":            2400,
        "protein":         1600,
        "phenotype":       1600,
        "similar_disease": 1800,
        "safety":          1200,
        "query_disease":   3200,
    }

    # Build layered positions
    role_groups: dict = defaultdict(list)
    for n, d in G.nodes(data=True):
        role_groups[d.get("role", "protein")].append(n)

    pos = {}
    for role, nodes in role_groups.items():
        x  = ROLE_X.get(role, 0.5)
        ys = np.linspace(0.93, 0.07, max(len(nodes), 2))
        for i, n in enumerate(sorted(nodes)):
            pos[n] = (x, float(ys[i]))

    # Draw
    fig, ax = plt.subplots(figsize=(22, 14))

    # Nodes
    for role in ROLE_COLOR:
        nodelist = [n for n, d in G.nodes(data=True)
                    if d.get("role") == role and n in pos]
        if not nodelist:
            continue
        nx.draw_networkx_nodes(
            G, pos, ax=ax, nodelist=nodelist,
            node_color=ROLE_COLOR[role],
            node_size=ROLE_SIZE.get(role, 1600),
            edgecolors="#555555", linewidths=1.3,
        )

    # Labels
    labels = {n: G.nodes[n].get("label", "") for n in G.nodes() if n in pos}
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=7.5)

    # Edges by role
    edge_styles = {
        "prediction": ("#1d4ed8", "solid",  2.8, True),
        "evidence":   ("#15803d", "solid",  1.5, True),
        "similarity": ("#7c3aed", "dashed", 2.0, True),
        "safety":     ("#dc2626", "dashed", 1.8, True),
    }
    for edge_role, (color, style, width, arrows) in edge_styles.items():
        el = [(u, v) for u, v, d in G.edges(data=True)
              if d.get("role") == edge_role and u in pos and v in pos]
        if el:
            nx.draw_networkx_edges(
                G, pos, ax=ax, edgelist=el,
                edge_color=color, style=style, width=width,
                arrows=arrows, arrowsize=14,
                min_source_margin=12, min_target_margin=12,
            )

    # Column headers
    for xf, header in [
        (0.05, "Candidate\nDrugs"),
        (0.35, "Protein\nEvidence"),
        (0.55, "Phenotype\nEvidence"),
        (0.72, "Similar\nDiseases"),
        (0.95, "Query\nDisease"),
    ]:
        ax.text(xf, 0.97, header, fontsize=10, fontweight="bold",
                ha="center", va="top", transform=ax.transAxes,
                color="#1e293b",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="#f1f5f9",
                          edgecolor="#94a3b8", linewidth=0.8))

    # Legend
    node_handles = [
        mpatches.Patch(facecolor=ROLE_COLOR[r], edgecolor="#555555",
                       label=r.replace("_", " ").title())
        for r in ["drug", "protein", "phenotype", "similar_disease",
                  "safety", "query_disease"]
    ]
    edge_handles = [
        mlines.Line2D([], [], color="#1d4ed8", linewidth=2.5,
                      label="Prediction"),
        mlines.Line2D([], [], color="#15803d", linewidth=1.8,
                      label="Evidence path"),
        mlines.Line2D([], [], color="#7c3aed", linewidth=2.0,
                      linestyle="dashed", label="Disease similarity"),
        mlines.Line2D([], [], color="#dc2626", linewidth=1.8,
                      linestyle="dashed", label="Safety flag"),
    ]
    leg1 = ax.legend(handles=node_handles, title="Node Type",
                     loc="lower left", bbox_to_anchor=(0.01, 0.01),
                     frameon=True, fancybox=True, framealpha=0.9,
                     fontsize=9, title_fontsize=10)
    ax.add_artist(leg1)
    ax.legend(handles=edge_handles, title="Edge Type",
              loc="lower right", bbox_to_anchor=(0.99, 0.01),
              frameon=True, fancybox=True, framealpha=0.9,
              fontsize=9, title_fontsize=10)

    ax.set_title(
        f"Drug Repurposing Evidence Graph — {dis_name}  (model: {model_name})\n"
        "Drugs  ·  Protein Evidence  ·  Phenotypes  ·  Similar Diseases  ·  Safety",
        fontsize=13, pad=22, color="#1e293b", fontweight="bold",
    )
    ax.axis("off")
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    print(f"  Subgraph saved → {save_path}")


# RAG Report builder

def _build_rag_report(
    dis_name:        str,
    dis_idx:         int,
    model_name:      str,
    similar_df:      pd.DataFrame,
    detailed_results:list,
    recommendations_df: pd.DataFrame,
    top_k_similar:   int = 5,
) -> tuple:
    """
    Build the human-readable RAG report from evidence + similarity data.
    Returns (final_report_str, rag_report_df, rag_prompt_packet_str).
    """

    # Similar disease summary block
    sim_lines = []
    for _, row in similar_df.head(top_k_similar).iterrows():
        sim_lines.append(
            f"{_safe(row['similar_disease_name'])} "
            f"(similarity={_fmt(row['combined_similarity_score'])}, "
            f"embedding={_fmt(row['embedding_cosine'])}, "
            f"shared proteins={int(row['shared_protein_count'])}, "
            f"shared effects={int(row['shared_effect_count'])})"
        )
    sim_summary = _bullets(sim_lines)

    # Per-drug explanation blocks
    drug_blocks = []
    report_rows = []

    for item in detailed_results:
        drug_name    = _safe(item["drug_name"])
        known_for    = _safe(item.get("known_treatment_for", ""))
        raw_score    = float(item["raw_model_score"])
        sim_support  = float(item.get("similar_disease_support", 0.0))
        rerank_score = float(item["rerank_score"])
        risk_level   = _safe(item["risk_level"])
        sup_dis      = _safe(item.get("supporting_similar_diseases", ""))

        bundle    = item["bundle"]
        ev_df     = item["evidence_df"].copy()
        sf_df     = item["safety_df"].copy()
        ev_counts = bundle.get("evidence_counts", {})

        # Sort evidence by strength priority
        if not ev_df.empty:
            ev_df["_pri"] = ev_df["evidence_type"].map(_ev_priority)
            ev_df = ev_df.sort_values(["_pri", "score_hint"], ascending=[True, False])
            ev_lines = [_humanise_evidence(r)
                        for _, r in ev_df.head(4).iterrows()]
        else:
            ev_lines = ["No strong graph evidence found for this candidate."]

        sf_lines = ([_humanise_safety(r) for _, r in sf_df.head(3).iterrows()]
                    if not sf_df.empty
                    else ["No direct contraindication or adverse overlap signal found."])

        block = f"""
Rank {item['rank']} — {drug_name}
{"─" * 60}
Currently known to treat : {known_for if known_for else "not found in training data"}
Raw model score          : {_fmt(raw_score)}
Similar disease support  : {_fmt(sim_support)}
Final rerank score       : {_fmt(rerank_score)}
Risk assessment          : {_risk_sentence(risk_level)}

Mechanistic evidence:
{_bullets(ev_lines)}

Evidence profile:
  • Shared protein bridges         : {ev_counts.get('shared_protein_bridge', 0)}
  • Protein interaction bridges    : {ev_counts.get('ppi_bridge', 0)}
  • Disease-to-disease transfer    : {ev_counts.get('disease_disease_transfer', 0)}
  • Phenotype / effect overlap     : {ev_counts.get('effect_or_phenotype_overlap', 0)}

Safety signals:
{_bullets(sf_lines)}

Support from similar diseases:
  {sup_dis if sup_dis.strip() else "No strong support identified from similar diseases."}
""".strip()

        drug_blocks.append(block)
        report_rows.append({
            "rank":                    item["rank"],
            "drug":                    drug_name,
            "known_treatment_for":     known_for,
            "model_score":             raw_score,
            "similar_disease_support": sim_support,
            "rerank_score":            rerank_score,
            "risk":                    risk_level,
            "evidence_summary":        "; ".join(ev_lines[:2]),
            "safety_summary":          "; ".join(sf_lines[:1]),
        })

    # Assemble full report
    sep = "\n" + "═" * 65 + "\n"
    intro = f"""
DRUG REPURPOSING REPORT
{"═" * 65}
Disease query   : {dis_name}
Model           : {model_name}

SIMILAR DISEASES USED AS CONTEXT
{"─" * 65}
Diseases most biologically similar to the query were identified using:
embedding cosine similarity · disease ontology edges · shared proteins
PPI-expanded protein bridges · phenotype overlap · shared therapeutics

{sim_summary}

TOP {len(drug_blocks)} CANDIDATE DRUGS
""".strip()

    footer = f"""

{"═" * 65}
INTERPRETATION NOTE
{"─" * 65}
These are graph-based research hypotheses, not clinical prescriptions.
The strongest candidates combine: high model score + strong mechanistic
evidence (especially shared protein bridges) + support from biologically
similar diseases + no direct contraindication signals.

Always validate predictions against current clinical literature before
drawing any experimental or clinical conclusions.
{"═" * 65}
""".strip()

    final_report = intro + sep + sep.join(drug_blocks) + "\n\n" + footer

    # Structured DataFrame version
    rag_df = pd.DataFrame(report_rows)

    # LLM-ready prompt packet
    rag_prompt = f"""You are given graph-based drug repurposing results for a disease query.

Task: Write a clear, human-understandable summary of the top drug candidates.
For each candidate:
  - Explain the mechanistic rationale (protein bridges, pathway connections)
  - Note the drug's current known uses (context for the repurposing hypothesis)
  - Mention any safety concerns from the graph
  - Quantify confidence using evidence type counts
  - Keep the tone cautious and non-clinical
  - State clearly these are graph-based research hypotheses

Disease: {dis_name}

Similar diseases (context):
{json.dumps(similar_df.head(top_k_similar)[["similar_disease_name","combined_similarity_score","shared_protein_count"]].to_dict("records"), indent=2, default=str)}

Top drug recommendations:
{json.dumps(recommendations_df[["drug_name","known_treatment_for","raw_model_score","rerank_score","risk_level","n_evidence"]].to_dict("records"), indent=2, default=str)}

Detailed evidence and safety per drug:
{json.dumps([item["rag_json"] for item in detailed_results], indent=2, default=str)[:12000]}
""".strip()

    return final_report, rag_df, rag_prompt


# Main run_demo() function

def run_demo(disease_query: str, model_choice: str = "hgt") -> dict:
    """
    Run the complete drug repurposing pipeline and display all four outputs.

    Parameters
    ----------
    disease_query : free-text disease name (e.g. "type 2 diabetes")
    model_choice  : "hgt" or "baseline"

    Returns
    -------
    dict containing all result objects for further inspection.

    Outputs displayed:
      1. Top 5 drug candidates table  (with Known Treatment For column)
      2. Top 10 similar diseases table
      3. Comprehensive evidence + similarity subgraph
      4. RAG explanation / summary report
    """

    # Model selection
    _c = model_choice.strip().lower()
    if _c == "hgt":
        assert "hgt_model" in globals(), "hgt_model not found — run Block 08."
        _model = hgt_model; _mname = "HGT"
    elif _c == "baseline":
        assert "baseline_model" in globals(), "baseline_model not found — run Block 07."
        _model = baseline_model; _mname = "Baseline"
    else:
        raise ValueError("model_choice must be 'hgt' or 'baseline'.")

    # Disease matching
    _match = match_disease(disease_query)
    assert len(_match) > 0, f"No disease found for '{disease_query}'."
    _dis_idx  = int(_match.iloc[0]["disease_idx"])
    _dis_name = str(_match.iloc[0]["disease_name"])

    print(f"\n{'═'*65}")
    print(f"  DRUG REPURPOSING DEMO")
    print(f"  Query   : '{disease_query}'  →  {_dis_name}")
    print(f"  Model   : {_mname}")
    print(f"{'═'*65}\n")

    # Raw retrieval + evidence reranking
    _W_EP, _W_PPI, _W_DD, _W_EFF = 3.0, 2.5, 2.0, 1.0
    _P_C,  _P_A,   _P_Z          = 4.0, 1.5, 1.5

    def _ev(b):
        c = b.get("evidence_counts", {})
        r = (_W_EP * c.get("shared_protein_bridge", 0) +
             _W_PPI * c.get("ppi_bridge", 0) +
             _W_DD  * c.get("disease_disease_transfer", 0) +
             _W_EFF * c.get("effect_or_phenotype_overlap", 0))
        return float(min(r / 10.0, 1.0))

    def _sf(b):
        rows = b.get("safety_rows", [])
        r = (_P_C * sum(1 for x in rows if x["safety_type"] == "direct_contraindication") +
             _P_A * sum(1 for x in rows if x["safety_type"] == "adverse_effect_overlap") +
             _P_Z * (1 if b["n_evidence_rows"] == 0 else 0))
        return float(min(r / 8.0, 1.0))

    _raw = score_all_drugs_for_disease(_model, graph, _dis_idx).copy()
    _known_train = TRAIN_THERAPEUTIC_BY_DISEASE.get(_dis_idx, set())
    _raw = _raw[~_raw["drug_idx"].isin(_known_train)].head(100).reset_index(drop=True)

    _r_rows = []; _r_bundles = []
    print(f"  Scoring {len(_raw)} drug candidates ...")
    for _, row in _raw.iterrows():
        did = int(row["drug_idx"]); rs = float(row["score"])
        b   = extract_evidence_bundle(did, _dis_idx, model=_model, top_n_each=5)
        evs = _ev(b); sfs = _sf(b)
        _r_rows.append({"drug_idx": did, "drug_name": b["drug_name"],
                         "raw_model_score": rs,
                         "rerank_score_base": float(0.65*rs + 0.30*evs - 0.20*sfs)})
        _r_bundles.append(b)

    _bundle_map = {int(b["drug_idx"]): b for b in _r_bundles}

    # Disease similarity
    print("  Computing disease similarity ...")

    _emb_cache = globals().get(f"{'hgt' if _c=='hgt' else 'baseline'}_embeddings")
    if isinstance(_emb_cache, dict) and SUP_DST_TYPE in _emb_cache:
        _emb = _emb_cache[SUP_DST_TYPE]
        _emb_mat = _emb.cpu().numpy() if torch.is_tensor(_emb) else np.asarray(_emb)
    else:
        with torch.no_grad():
            _z = _model.encode(graph.to(next(_model.parameters()).device))
        _emb_mat = _z[SUP_DST_TYPE].detach().cpu().numpy()

    _q_emb  = _emb_mat[_dis_idx]
    _q_pr   = _build_sets(SUP_DST_TYPE, disease_protein_edge_types).get(_dis_idx, set())
    _q_ef   = _build_sets(SUP_DST_TYPE, disease_effect_edge_types).get(_dis_idx, set())
    _q_dd   = _build_sets(SUP_DST_TYPE, disease_disease_edge_types).get(_dis_idx, set())
    _q_dr   = TRAIN_THERAPEUTIC_BY_DISEASE.get(_dis_idx, set())
    _q_prx  = _ppi_expand(_q_pr)

    _DIS_PR = _build_sets(SUP_DST_TYPE, disease_protein_edge_types)
    _DIS_EF = _build_sets(SUP_DST_TYPE, disease_effect_edge_types)
    _DIS_DD = _build_sets(SUP_DST_TYPE, disease_disease_edge_types)

    _sim_rows = []
    for cand in range(int(graph[SUP_DST_TYPE].num_nodes)):
        if cand == _dis_idx: continue
        c_pr = _DIS_PR.get(cand, set())
        c_ef = _DIS_EF.get(cand, set())
        c_dd = _DIS_DD.get(cand, set())
        c_dr = TRAIN_THERAPEUTIC_BY_DISEASE.get(cand, set())
        cos  = float(np.dot(_q_emb, _emb_mat[cand]) /
                     max(np.linalg.norm(_q_emb) * np.linalg.norm(_emb_mat[cand]), 1e-12))
        _sim_rows.append({
            "similar_disease_idx":           cand,
            "similar_disease_name":          get_node_name(SUP_DST_TYPE, cand),
            "embedding_cosine":              cos,
            "direct_disease_disease_edge":   int((SUP_DST_TYPE, cand) in _q_dd or
                                                  (SUP_DST_TYPE, _dis_idx) in c_dd),
            "shared_protein_count":          len(_q_pr & c_pr),
            "ppi_bridge_count":              len((_q_prx & c_pr) - _q_pr),
            "shared_effect_count":           len(_q_ef & c_ef),
            "train_drug_overlap_count":      len(_q_dr & c_dr),
            "similar_known_train_drug_count": len(c_dr),
            "sample_similar_train_drugs": ", ".join(
                _NODE_NAME_LOOKUP.get((SUP_SRC_TYPE.lower(), i), f"drug_{i}")
                for i in list(sorted(c_dr))[:3]
            ),
        })

    _sim_df = pd.DataFrame(_sim_rows)
    _arr = lambda col: (_log_norm(_sim_df[col].values)
                        if _sim_df[col].max() > 0 else np.zeros(len(_sim_df)))
    _sim_df["combined_similarity_score"] = (
        0.35 * ((_sim_df["embedding_cosine"].clip(-1, 1) + 1) / 2) +
        0.20 * _sim_df["direct_disease_disease_edge"] +
        0.20 * _arr("shared_protein_count") +
        0.10 * _arr("ppi_bridge_count") +
        0.10 * _arr("shared_effect_count") +
        0.05 * _arr("train_drug_overlap_count")
    )
    _sim_top = (_sim_df.sort_values("combined_similarity_score", ascending=False)
                       .head(10).reset_index(drop=True))

    # Disease-support scores for drugs
    _sup = defaultdict(float); _sup_d = defaultdict(list)
    for _, sr in _sim_top.head(20).iterrows():
        si = int(sr["similar_disease_idx"]); ss = float(sr["combined_similarity_score"])
        sn = str(sr["similar_disease_name"])
        for d in TRAIN_THERAPEUTIC_BY_DISEASE.get(si, set()):
            _sup[d] += ss
            if len(_sup_d[d]) < 5:
                _sup_d[d].append(sn)

    _max_s = max(_sup.values()) if _sup else 1.0

    # Final reranking with disease support
    _final_rows = []
    for row_d in _r_rows:
        did   = int(row_d["drug_idx"]); rs = float(row_d["raw_model_score"])
        b     = _bundle_map[did]
        s_n   = float(_sup.get(did, 0.0)) / _max_s
        s_dis = " | ".join(_sup_d.get(did, []))
        fs    = 0.55*rs + 0.20*_ev(b) + 0.25*s_n - 0.15*_sf(b)
        _final_rows.append({
            "drug_idx":                   did,
            "drug_name":                  b["drug_name"],
            "known_treatment_for":        get_known_treatments(did),
            "raw_model_score":            rs,
            "similar_disease_support":    s_n,
            "supporting_similar_diseases": s_dis,
            "rerank_score":               float(fs),
            "risk_level":                 b["risk_level"],
            "n_evidence":                 b["n_evidence_rows"],
            "n_safety":                   b["n_safety_flags"],
        })

    _final_df = (pd.DataFrame(_final_rows)
                   .sort_values("rerank_score", ascending=False)
                   .head(5).reset_index(drop=True))

    # Detailed results list
    _det = []
    for rank, (_, rr) in enumerate(_final_df.iterrows(), start=1):
        did = int(rr["drug_idx"]); b = _bundle_map[did]
        ev_df2, sf_df2 = evidence_bundle_to_tables(b)
        _det.append({
            "rank":                    rank,
            "drug_idx":                did,
            "drug_name":               b["drug_name"],
            "known_treatment_for":     str(rr["known_treatment_for"]),
            "raw_model_score":         float(rr["raw_model_score"]),
            "similar_disease_support": float(rr["similar_disease_support"]),
            "supporting_similar_diseases": str(rr["supporting_similar_diseases"]),
            "rerank_score":            float(rr["rerank_score"]),
            "risk_level":              b["risk_level"],
            "n_evidence":              b["n_evidence_rows"],
            "n_safety":                b["n_safety_flags"],
            "bundle":                  b,
            "evidence_df":             ev_df2,
            "safety_df":               sf_df2,
            "rag_json":                bundle_to_rag_json(b),
        })

    # OUTPUT 1 — Top 5 Drug Candidates Table
    print(f"\n{'═'*65}")
    print(f"  OUTPUT 1 — TOP 5 CANDIDATE DRUGS")
    print(f"  Disease : {_dis_name}  |  Model : {_mname}")
    print(f"{'═'*65}")
    display(
        _final_df[[
            "drug_name",
            "raw_model_score", "similar_disease_support",
            "rerank_score", "risk_level", "n_evidence", "n_safety",
            "known_treatment_for"
        ]].rename(columns={
            "drug_name":               "Drug",
            "raw_model_score":         "Model Score",
            "similar_disease_support": "Sim. Support",
            "rerank_score":            "Rerank Score",
            "risk_level":              "Risk",
            "n_evidence":              "Evidence",
            "n_safety":                "Safety Flags",
            "known_treatment_for":     "Known Treatment For",
        })
        .assign(Rank=range(1, 6))[
            ["Rank", "Drug", "Model Score",
             "Sim. Support", "Rerank Score", "Risk", "Evidence", "Safety Flags",
             "Known Treatment For"]
        ].set_index("Rank")
    )

    # OUTPUT 2 — Similar Diseases Table
    print(f"\n{'═'*65}")
    print(f"  OUTPUT 2 — TOP 10 SIMILAR DISEASES")
    print(f"{'═'*65}")
    display(
        _sim_top[[
            "similar_disease_name", "combined_similarity_score",
            "embedding_cosine", "shared_protein_count",
            "shared_effect_count", "train_drug_overlap_count"
        ]].rename(columns={
            "similar_disease_name":      "Similar Disease",
            "combined_similarity_score": "Similarity Score",
            "embedding_cosine":          "Embedding Cos",
            "shared_protein_count":      "Shared Proteins",
            "shared_effect_count":       "Shared Effects",
            "train_drug_overlap_count":  "Shared Drugs",
        })
        .assign(Rank=range(1, len(_sim_top)+1))
        .set_index("Rank")
    )

    # OUTPUT 3 — Evidence + Similarity Subgraph
    print(f"\n{'═'*65}")
    print(f"  OUTPUT 3 — EVIDENCE & SIMILARITY SUBGRAPH")
    print(f"{'═'*65}")

    _G = _build_demo_subgraph(
        dis_idx=_dis_idx,
        dis_name=_dis_name,
        detailed_results=_det,
        similar_diseases=_sim_top,
        top_k_drugs=5,
        top_k_similar=5,
    )
    _draw_demo_subgraph(_G, _dis_name, _mname,
                        save_path="/mnt/data/demo_subgraph.png")

    # OUTPUT 4 — RAG Explanation / Summary
    print(f"\n{'═'*65}")
    print(f"  OUTPUT 4 — RAG EXPLANATION & SUMMARY")
    print(f"{'═'*65}\n")

    _report, _rag_df, _rag_prompt = _build_rag_report(
        dis_name=_dis_name,
        dis_idx=_dis_idx,
        model_name=_mname,
        similar_df=_sim_top,
        detailed_results=_det,
        recommendations_df=_final_df,
        top_k_similar=5,
    )
    print(_report)

    print(f"\n{'─'*65}")
    print("  Structured RAG report table:")
    display(_rag_df.set_index("rank"))

    return {
        "disease_query":               disease_query,
        "selected_model_name":         _mname,
        "selected_disease_idx":        _dis_idx,
        "selected_disease_name":       _dis_name,
        "match_candidates_df":         _match,
        "query_similar_diseases_df":   _sim_top,
        "query_drug_recommendations_df": _final_df,
        "query_detailed_results":      _det,
        "subgraph_nx":                 _G,
        "rag_report_df":               _rag_df,
        "final_human_report":          _report,
        "rag_prompt_packet":           _rag_prompt,
    }


# Summary
print(f"\n{'='*55}")
print(f"  BLOCK 13 COMPLETE")
print(f"{'='*55}")
print("  run_demo(disease_query, model_choice='hgt') is now ready.")
print("  Run Block 14 to execute the demo.")
print(f"{'='*55}")


  BLOCK 13 COMPLETE
  run_demo(disease_query, model_choice='hgt') is now ready.
  Run Block 14 to execute the demo.
